# 2. LLM Triage, Policy RAG and Refund Agent — Parts C and D

Run notebook 1 first, then **Run All** here. Use the same Python environment. Every model call uses the supplied offline `mock_llm.py`; no account, key, paid service or network is used. The mock simulates classifier formatting failures, transient errors, grounded sentence selection and tool requests; it is not a hosted general-purpose LLM.

This notebook regenerates triage/evaluation outputs and resets the synthetic `demo_*` agent runs, ledger and checkpoints through the repeatable benchmark function. No real refund is issued. Do not use these simulated policies or probabilities for real customer decisions. The canonical implementation remains in `src/` and the required report in README.


## 2.1 Locate the project and verify prerequisites


In [1]:
from pathlib import Path
import os, sys, json, runpy
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '2')
start = Path.cwd().resolve()
candidates = [start, *start.parents, start / 'instamart-copilot']
ROOT = next((p for p in candidates if (p / 'run_project.py').is_file() and (p / 'src').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the instamart-copilot project folder or its notebooks subfolder.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
OUT = ROOT / 'outputs'
def report(name):
    return json.loads((OUT / name).read_text(encoding='utf-8'))
print('Python:', sys.version.split()[0])
print('Project:', ROOT.name)


Python: 3.12.14
Project: instamart-copilot


In [2]:
required = ['data/raw/tickets.csv', 'data/raw/policy_eval.csv', 'data/raw/agent_scenarios.csv',
            'data/instamart.db', 'outputs/customer_segments.csv', 'outputs/lapse_model.joblib']
missing = [name for name in required if not (ROOT / name).is_file()]
if missing:
    raise FileNotFoundError('Run notebook 1 first. Missing: ' + ', '.join(missing))
import mock_llm
print('All notebook 1 prerequisites exist; offline mock loaded.')


All notebook 1 prerequisites exist; offline mock loaded.


## 2.2 Guardrails and privacy
Empty text is never sent to the model, and text over 400 estimated tokens is flagged. Regexes redact email, phone, then order IDs in exactly that order. Check that the demonstration contains placeholders only after processing. `est_tokens` is the supplied ceil(words×1.3), not an actual tokenizer.


In [3]:
from src.triage import redact, validate_output, ZERO, FEW, call_with_retry, run_all as run_triage
example_ticket = 'I was charged twice for ORD-123456. Email user@example.com or call 9876543210.'
safe_ticket, redaction_counts = redact(example_ticket)
print('Redacted ticket:', safe_ticket)
print('Substitutions:', redaction_counts)
tickets = pd.read_csv(ROOT / 'data/raw/tickets.csv', keep_default_na=False)
print('Ticket rows:', len(tickets))
print('Empty tickets:', int(tickets.text.str.strip().eq('').sum()))
print('Long-ticket estimates:', [mock_llm.est_tokens(t) for t in tickets.text if mock_llm.est_tokens(t) > 400])


Redacted ticket: I was charged twice for [ORDER_ID]. Email [EMAIL] or call [PHONE].
Substitutions: {'email': 1, 'phone': 1, 'order': 1}
Ticket rows: 164
Empty tickets: 4
Long-ticket estimates: [734, 734, 734, 734]


## 2.3 Prompts and strict output validation
Few-shot examples improve the mock's category priority. Valid JSON with a disallowed category is rejected; non-JSON is accepted only when exactly one distinct allowed category word is found using regex word extraction.


In [4]:
print('ZERO-SHOT SYSTEM PROMPT:\n', ZERO)
print('\nFEW-SHOT SYSTEM PROMPT:\n', FEW)
for response in ['{"category":"fraud","reason":"unauthorized"}',
                 'The customer appears to have a billing-related problem.',
                 '{"category":"shipping","reason":"billing"}', 'billing and fraud']:
    print(response, '->', validate_output(response))


ZERO-SHOT SYSTEM PROMPT:
 CLASSIFY support tickets into billing, delivery, technical, fraud. Reply JSON only: {"category": "...", "reason": "..."}. Choose the category matching the customer problem.

FEW-SHOT SYSTEM PROMPT:
 CLASSIFY support tickets into billing, delivery, technical, fraud. Reply JSON only: {"category": "...", "reason": "..."}. Choose the category matching the customer problem.
EXAMPLES:
Unauthorized payment on my card: {"category":"fraud","reason":"unauthorized transaction"}
The app freezes on payment: {"category":"technical","reason":"app error"}
Delivery was late: {"category":"delivery","reason":"late delivery"}
Wrong invoice amount: {"category":"billing","reason":"incorrect bill"}
{"category":"fraud","reason":"unauthorized"} -> ('classified_json', 'fraud')
The customer appears to have a billing-related problem. -> ('classified_keyword', 'billing')
{"category":"shipping","reason":"billing"} -> ('flagged_invalid_output', '')
billing and fraud -> ('flagged_invalid_out

## 2.4 Compare full runs and enforce the token budget
Each independent run resets the mock. Calls retry at most three times with exponential jittered backoff and log each retry. The zero/few-shot comparisons have no budget; the final few-shot output uses 100000 tokens. A zero-shot budget of 6000 must exhaust at least one ticket. This cell repeats four benchmark runs; retry lines are expected.


In [5]:
run_triage()
triage_summary = report('part_c_summary.json')
rows = []
for name, result in triage_summary.items():
    rows.append({'run':name, 'classified':result['classified'], 'correct':result['correct'],
                 'accuracy':result['accuracy'], 'total_tokens':result['total_tokens'],
                 'tokens_per_classified':result['tokens_per_classified'], 'retries':result['retries'],
                 'budget_exhausted':result['status_counts'].get('budget_exhausted', 0)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nStatus counts:', triage_summary['few']['status_counts'])
print('PII substitutions:', triage_summary['few']['redactions'])
print('\nZero-shot confusion table:\n', pd.read_csv(OUT / 'triage_zero_confusion.csv').to_string(index=False))


T0002 retry 0: wait 0.052993s
T0116 retry 0: wait 0.045313s
T0153 retry 0: wait 0.048438s
T0039 retry 0: wait 0.047790s
T0152 retry 0: wait 0.054206s
T0043 retry 0: wait 0.053459s
T0111 retry 0: wait 0.056152s
T0155 retry 0: wait 0.046087s
T0100 retry 0: wait 0.050274s
T0068 retry 0: wait 0.045372s
T0135 retry 0: wait 0.047733s
T0009 retry 0: wait 0.051317s
T0067 retry 0: wait 0.045332s
T0106 retry 0: wait 0.047485s
T0097 retry 0: wait 0.053124s
T0003 retry 0: wait 0.051812s
T0019 retry 0: wait 0.047756s
T0086 retry 0: wait 0.052366s
T0065 retry 0: wait 0.055118s
T0002 retry 0: wait 0.052993s
T0116 retry 0: wait 0.045313s
T0153 retry 0: wait 0.048438s
T0039 retry 0: wait 0.047790s
T0152 retry 0: wait 0.054206s
T0043 retry 0: wait 0.053459s
T0111 retry 0: wait 0.056152s
T0155 retry 0: wait 0.046087s
T0100 retry 0: wait 0.050274s
T0068 retry 0: wait 0.045372s
T0135 retry 0: wait 0.047733s
T0009 retry 0: wait 0.051317s
T0067 retry 0: wait 0.045332s
T0106 retry 0: wait 0.047485s
T0097 retr

## 2.5 Routing and human review
Guardrail, invalid-output and budget cases go to human_review with NORMAL priority. Otherwise fraud goes to security_team HIGH; non-fraud Premium customers go to premium_desk HIGH; others go to support_queue NORMAL. The Premium flag is read from notebook 1's saved segments. Rule validation checks allowed output structure/category, not real-world truth.


In [6]:
triage_results = pd.read_csv(OUT / 'triage_results.csv', keep_default_na=False)
print('Final rows:', len(triage_results))
print(triage_results.groupby(['route','priority']).size().to_string())
print('\nFirst eight results:\n', triage_results.head(8).to_string(index=False))
assert len(triage_results) == 164


Final rows: 164
route          priority
human_review   NORMAL       17
premium_desk   HIGH         11
security_team  HIGH         34
support_queue  NORMAL      102

First eight results:
 ticket_id customer_id          status  category segment_name priority         route  tokens_used
    T0024       C0486 classified_json     fraud      Dormant     HIGH security_team           94
    T0073       C0080 classified_json   billing     Frequent   NORMAL support_queue           90
    T0095       C1087 classified_json technical   Occasional   NORMAL support_queue           92
    T0159       C1172   skipped_empty               Frequent   NORMAL  human_review            0
    T0080       C0910 classified_json     fraud   Occasional     HIGH security_team           94
    T0124       C0139 classified_json     fraud     Frequent     HIGH security_team           94
    T0061       C0812 classified_json   billing     Frequent   NORMAL support_queue           86
    T0118       C0355 classified_json

## 2.6 Try a new support ticket
Edit `new_ticket_text`, then rerun this cell. The model sees exactly the redacted text. This example classifies only; it does not add a customer to the database or permanently reroute a real support case.


In [7]:
import random
new_ticket_text = 'Someone hacked my account and used my wallet balance. Email test@example.com.'
if not new_ticket_text.strip():
    print('skipped_empty — human_review')
elif mock_llm.est_tokens(new_ticket_text) > 400:
    print('flagged_long — human_review')
else:
    safe_text, counts = redact(new_ticket_text)
    mock_llm.reset()
    response = call_with_retry([{'role':'system','content':FEW}, {'role':'user','content':safe_text}],
                               'DEMO_TICKET', [], random.Random(42))
    status, category = validate_output(response.choices[0].message.content)
    print('Redacted input:', safe_text)
    print('Status:', status, 'Category:', category, 'Tokens:', response.usage.total_tokens)


Redacted input: Someone hacked my account and used my wallet balance. Email [EMAIL]
Status: classified_json Category: fraud Tokens: 84


## 2.7 Chunk, vectorize and retrieve policies
Forty-word chunks overlap by ten words (stride 30); stop when the first chunk reaches the document end. TF measures term frequency, IDF downweights common terms, and cosine similarity ranks query/chunk vectors. Different phrasings with the same meaning can still fail to match.


In [8]:
from collections import Counter
from src.policy_rag import PolicyRAG, run_all as run_rag
rag = PolicyRAG(overlap=10)
print('Chunk count:', len(rag.chunks))
print('Chunks per source:', dict(Counter(c['source'] for c in rag.chunks)))
for hit in rag.retrieve('refund policy late delivery', k=3):
    print('\nSource:', hit['source'], 'Score:', round(hit['score'], 4))
    print(hit['text'])
assert len(rag.chunks) == 15


Chunk count: 15
Chunks per source: {'delivery_policy.txt': 3, 'escalation_policy.txt': 3, 'payment_policy.txt': 3, 'privacy_policy.txt': 2, 'refund_policy.txt': 4}

Source: refund_policy.txt Score: 0.2719
order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are

Source: refund_policy.txt Score: 0.2514
Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

Source: refund_policy.txt Score: 0.2454
late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR 1000 must be approved by a human support lead before money is returned. Approved refunds are credited to the original


## 2.8 Ask a policy question
Edit the question. The exact CONTEXT/QUESTION message includes the top three chunks with source labels; the offline POLICY ASSISTANT returns a context-derived sentence or a fallback.


In [9]:
new_question = 'Who must approve a refund above INR 1000?'
policy_answer = rag.answer(new_question)
print('Question:', new_question)
print('Answer:', policy_answer['answer'])
print('Sources:', policy_answer['sources'])
print('Tokens:', policy_answer['tokens_used'])
print('\nUnanswerable example:', rag.answer('Who won the cricket world cup final?')['answer'])


Question: Who must approve a refund above INR 1000?
Answer: A refund above INR 1000 must be approved by a human support lead before money is returned.
Sources: ['refund_policy.txt', 'refund_policy.txt', 'refund_policy.txt']
Tokens: 222

Unanswerable example: I could not find that information in the policy documents.


## 2.9 Evaluate grounding and remove overlap
For twelve answerable questions, count top-three expected-document retrieval and case-sensitive expected-fact matches. The other two should start with the fallback. Zero overlap loses boundary-spanning facts: the late-refund condition is separated from its conclusion, and the failed-payment sentence is separated from "within 48 hours".


In [10]:
run_rag()
rag_summary = report('part_d_rag_summary.json')
print(pd.DataFrame([{'overlap':overlap, **{k:result[k] for k in ['chunks','retrieval_hits','fact_hits','fallbacks']}}
                    for overlap,result in rag_summary.items()]).to_string(index=False))
print('\nQuestions broken with no overlap:', rag_summary['0']['failed_questions'])
zero_overlap = pd.read_csv(OUT / 'policy_eval_overlap_0.csv', keep_default_na=False)
print(zero_overlap[zero_overlap.question.isin(rag_summary['0']['failed_questions'])][['question','expected_fact','answer']].to_string(index=False))


overlap  chunks  retrieval_hits  fact_hits  fallbacks
     10      15              12         12          2
      0      13              12         10          2

Questions broken with no overlap: ['What delivery time makes a completed order eligible for a full refund?', 'How soon is money returned when a payment fails but is deducted?']
                                                              question        expected_fact                                                                                                                                                     answer
What delivery time makes a completed order eligible for a full refund? more than 45 minutes                                                                                                                receives a full refund of the order amount.
      How soon is money returned when a payment fails but is deducted?             48 hours If a payment fails but money is deducted, the amount is returned automatic

## 2.10 Tools and refund rules
The tool specifications declare names, descriptions and parameter schemas; TOOL_MAP dispatches calls. Read-only lookup uses parameterized SQL. Only completed orders delivered in more than 45 minutes qualify for their full order amount. Up to INR 1000 is automatically recorded; larger refunds are escalated without issuing money. Issuing tools independently recheck eligibility and amount. Requests are treated as within seven days.


In [11]:
from src.refund_agent import TOOLS_SPEC, TOOL_MAP, get_order, calculate_refund, run_all as run_refund_demos
print(json.dumps(TOOLS_SPEC, indent=2))
print('Available implementations:', sorted(TOOL_MAP))
print('At 45 minutes:', calculate_refund(100, 45))
print('At 46 minutes:', calculate_refund(100, 46))
print('Unknown order:', get_order('ORD-999999'))


[
  {
    "name": "get_order",
    "description": "Look up a final order in the clean SQLite database.",
    "parameters": {
      "type": "object",
      "properties": {
        "order_id": {
          "type": "string"
        }
      },
      "required": [
        "order_id"
      ],
      "additionalProperties": false
    }
  },
  {
    "name": "search_policy",
    "description": "Retrieve the three most similar policy chunks.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string"
        }
      },
      "required": [
        "query"
      ],
      "additionalProperties": false
    }
  },
  {
    "name": "calculate_refund",
    "description": "Calculate the full amount for delivery over 45 minutes, otherwise zero.",
    "parameters": {
      "type": "object",
      "properties": {
        "order_amount": {
          "type": "number"
        },
        "delivery_minutes": {
          "type": [
            "number",
         

## 2.11 ReAct scenarios, three limits and crash recovery
**This cell resets the benchmark's synthetic ledger/run logs and demo checkpoints.** The agent records assistant tool calls and JSON observations, then loops until a final answer or limit. It checks monotonic timeout before each call and token usage immediately after the response, before running a requested tool. Six unfinished calls produce a max-iterations fallback. Every tool observation saves an atomic checkpoint. S1 crashes after tool two, then resumes without repeating tools; ledger execution keys guard replay within the run.


In [12]:
run_refund_demos()
agent_summary = report('part_d_agent_summary.json')
print(pd.DataFrame(agent_summary['scenarios']).to_string(index=False))
print('\nToken-limit demonstration:', agent_summary['token_demo'])
print('Zero-second timeout demonstration:', agent_summary['timeout_demo'])
print('Crash/resume tools:', ' -> '.join(agent_summary['resume_tools']))
print('Resumed status:', agent_summary['resumed_status'])
checkpoint = json.loads((ROOT / 'state/demo_crash.json').read_text(encoding='utf-8'))
print('Resumed refund INR:', checkpoint['steps'][-1]['observation']['refund_inr'])
print('Executed tools:', checkpoint['tools_executed'])
assert checkpoint['tools_executed'] == ['get_order','search_policy','calculate_refund','issue_refund']
assert checkpoint['steps'][-1]['observation']['refund_inr'] == 128.25


scenario_id                  status  llm_calls  tokens_used  refund_inr
         S1                  REFUND          5          952      128.25
         S2                  REFUND          5          952      135.00
         S3                ESCALATE          5          939     1999.00
         S4                  DENIED          4          649        0.00
         S5                  DENIED          3          370        0.00
         S6               NOT_FOUND          2           94        0.00
         S7 FALLBACK_MAX_ITERATIONS          6         3066        0.00

Token-limit demonstration: FALLBACK_TOKEN_LIMIT
Zero-second timeout demonstration: FALLBACK_TIMEOUT
Crash/resume tools: get_order -> search_policy -> calculate_refund -> issue_refund
Resumed status: REFUND
Resumed refund INR: 128.25
Executed tools: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund']


### ReAct versus plan-and-execute
S1 needs five calls, a cancelled order three, and NOT_FOUND two. ReAct repeatedly sends history but adapts to each observation. A validated plan with deterministic branch rules could use one planning call and replan only on unexpected results. This small fixed refund workflow favours deterministic execution of a validated plan, with the same independent eligibility checks, limits and checkpoints. Neither approach substitutes for genuine payment authorization or production transaction guarantees.


## 2.12 Run the complete checks and refresh the README
The original required scripts remain unmodified. The README reports all grading numbers, limitations and source-file differences; these notebooks are optional code walkthroughs.


In [13]:
from src.report import write_readme
write_readme()
import unittest
suite = unittest.defaultTestLoader.discover(str(ROOT / 'tests'))
test_result = unittest.TextTestRunner(stream=sys.stdout, verbosity=1).run(suite)
assert test_result.wasSuccessful()
print('LLM NOTEBOOK COMPLETE')


....................test retry 0: wait 0.052993s
test retry 1: wait 0.090625s
test retry 2: wait 0.193751s
.
----------------------------------------------------------------------
Ran 21 tests in 0.814s

OK
LLM NOTEBOOK COMPLETE
